# 06.9 · Market basket analysis: support, confidence, lift

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/09-market-basket/lesson.ipynb)

Read the [README](README.md) first and fill in the 8-basket table on paper. This notebook checks your numbers, then mines a year of ShopWave's French orders for products that sell together.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

## 1. Eight baskets

In [2]:
baskets = [
    ["tea", "cake", "mug"],
    ["tea", "cake"],
    ["tea", "mug"],
    ["cake", "candle"],
    ["tea", "cake", "card"],
    ["candle", "card"],
    ["tea", "cake", "candle"],
    ["mug", "card"],
]
items = sorted({i for b in baskets for i in b})
onehot = pd.DataFrame([[i in b for i in items] for b in baskets], columns=items)
onehot.index = [f"B{k}" for k in range(1, 9)]
onehot.astype(int)

,cake,candle,card,mug,tea
B1,1,0,0,1,1
B2,1,0,0,0,1
B3,0,0,0,1,1
B4,1,1,0,0,0
B5,1,0,1,0,1
B6,0,1,1,0,0
B7,1,1,0,0,1
B8,0,0,1,1,0


## 2. Support, confidence, lift by hand

Each line below is one formula from the README.

In [3]:
def support(*things):
    return onehot[list(things)].all(axis=1).mean()

def confidence(a, b):
    return support(a, b) / support(a)

def lift(a, b):
    return confidence(a, b) / support(b)

for a, b in [("tea", "cake"), ("mug", "tea"), ("candle", "card")]:
    print(f"{a:>6} -> {b:<5}  support {support(a, b):.3f}   confidence {confidence(a, b):.3f}   lift {lift(a, b):.3f}")

   tea -> cake   support 0.500   confidence 0.800   lift 1.280
   mug -> tea    support 0.250   confidence 0.667   lift 1.067
candle -> card   support 0.125   confidence 0.333   lift 0.889


- **tea → cake**: lift 1.28. People who buy tea buy cake 28% more often than shoppers in general.
- **mug → tea**: lift 1.07, barely above 1. Tea is popular anyway, so this rule tells you little.
- **candle → card**: lift 0.89, *below* 1. Candle buyers are slightly *less* likely than average to buy a card.

## 3. The same with mlxtend

`apriori` finds every itemset whose support is above a minimum; `association_rules` turns them into rules.

In [4]:
from mlxtend.frequent_patterns import apriori, association_rules

def basket_matrix(df, drop_postage=True):
    """One row per invoice, one True/False column per product."""
    b = df.groupby(["Invoice", "Description"])["Quantity"].sum().unstack(fill_value=0) > 0
    if drop_postage:
        b = b.drop(columns=[c for c in b.columns if "POSTAGE" in str(c)])
    return b

def show(rules, n=8):
    out = rules.sort_values("lift", ascending=False).head(n).copy()
    out["if bought"] = out["antecedents"].apply(lambda s: " + ".join(sorted(s)))
    out["then also"] = out["consequents"].apply(lambda s: " + ".join(sorted(s)))
    return out[["if bought", "then also", "support", "confidence", "lift"]].round(3)

frequent = apriori(onehot, min_support=0.2, use_colnames=True)
print(frequent.assign(itemsets=frequent["itemsets"].apply(lambda s: ", ".join(sorted(s)))))

   support      itemsets
0    0.625          cake
1    0.375        candle
2    0.375          card
3    0.375           mug
4    0.625           tea
5    0.250  cake, candle
6    0.500     cake, tea
7    0.250      mug, tea


In [5]:
rules = association_rules(frequent, metric="lift", min_threshold=0)
show(rules, n=10)

,if bought,then also,support,confidence,lift
2,cake,tea,0.50,0.800,1.280
3,tea,cake,0.50,0.800,1.280
0,cake,candle,0.25,0.400,1.067
1,candle,cake,0.25,0.667,1.067
4,mug,tea,0.25,0.667,1.067
5,tea,mug,0.25,0.400,1.067


The tea/cake rules match the hand calculation. Candle/card is missing: with support 1/8 = 0.125 it is below the 0.2 minimum, so Apriori never even considered it.

**The Apriori principle**: if {candle} is not frequent, then {candle, card} cannot be either, because every basket containing both also contains candle. So Apriori only builds bigger itemsets out of smaller ones that already passed. This is what makes it fast enough for thousands of products.

## 4. ShopWave: a year of French orders

Restricting to one country keeps the basket matrix small (one row per invoice, one column per product).

In [6]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"], **{"Customer ID": sales["Customer ID"].astype(int)})
print(f"{len(sales):,} clean transaction lines, {sales['Customer ID'].nunique():,} customers")

392,693 clean transaction lines, 4,338 customers


In [7]:
france = basket_matrix(sales[sales["Country"] == "France"])
print(f"{france.shape[0]} invoices, {france.shape[1]} different products")
print(f"average basket: {france.sum(axis=1).mean():.1f} products")

389 invoices, 1543 different products
average basket: 20.6 products


In [8]:
frequent_fr = apriori(france, min_support=0.05, use_colnames=True)
rules_fr = association_rules(frequent_fr, metric="lift", min_threshold=1.0)
print(f"{len(frequent_fr)} frequent itemsets, {len(rules_fr)} rules with lift > 1")
show(rules_fr, n=10)

112 frequent itemsets, 86 rules with lift > 1


,if bought,then also,support,confidence,lift
38,PACK OF 6 SKULL PAPER PLATES,PACK OF 6 SKULL PAPER CUPS,0.051,0.909,14.145
39,PACK OF 6 SKULL PAPER CUPS,PACK OF 6 SKULL PAPER PLATES,0.051,0.800,14.145
9,CHILDRENS CUTLERY SPACEBOY,CHILDRENS CUTLERY DOLLY GIRL,0.064,0.926,12.864
8,CHILDRENS CUTLERY DOLLY GIRL,CHILDRENS CUTLERY SPACEBOY,0.064,0.893,12.864
71,ALARM CLOCK BAKELIKE RED,ALARM CLOCK BAKELIKE GREEN + ALARM CLOCK BAKEL...,0.064,0.676,9.063
70,ALARM CLOCK BAKELIKE GREEN + ALARM CLOCK BAKEL...,ALARM CLOCK BAKELIKE RED,0.064,0.862,9.063
72,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE PINK + ALARM CLOCK BAKELI...,0.064,0.658,8.825
69,ALARM CLOCK BAKELIKE PINK + ALARM CLOCK BAKELI...,ALARM CLOCK BAKELIKE GREEN,0.064,0.862,8.825
2,ALARM CLOCK BAKELIKE RED,ALARM CLOCK BAKELIKE GREEN,0.080,0.838,8.577
3,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED,0.080,0.816,8.577


Read the top rules. They are mostly **sets**: skull paper plates with skull paper cups, the Spaceboy children's cutlery with the Dolly Girl one, alarm clocks in several colours. Customers buy a matching set, or several colours of the same product. That is useful:

- **Website**: "complete the set" suggestions.
- **Warehouse**: store set members next to each other.
- **Promotions**: discount one item to sell the other, but only if lift is high *and* the pair is not bought together anyway (high confidence already means little extra to gain).

## 5. Pitfall: popular items make high-confidence, useless rules

Above we dropped `POSTAGE`. French customers pay postage on almost every order. Put it back:

In [9]:
france_post = basket_matrix(sales[sales["Country"] == "France"], drop_postage=False)
print(f"POSTAGE appears in {france_post['POSTAGE'].mean():.0%} of French invoices")
rules_post = association_rules(apriori(france_post, min_support=0.05, use_colnames=True), metric="confidence", min_threshold=0.8)
to_postage = rules_post[rules_post["consequents"] == frozenset({"POSTAGE"})]
print(f"{len(to_postage)} rules of the form 'X -> POSTAGE' with confidence >= 0.8")
show(to_postage.sort_values("confidence", ascending=False), n=5)

POSTAGE appears in 77% of French invoices
69 rules of the form 'X -> POSTAGE' with confidence >= 0.8


,if bought,then also,support,confidence,lift
74,PLASTERS IN TIN CIRCUS PARADE + RED TOADSTOOL...,POSTAGE,0.051,1.000,1.292
76,PLASTERS IN TIN WOODLAND ANIMALS + RED TOADSTO...,POSTAGE,0.054,1.000,1.292
19,JUMBO BAG WOODLAND ANIMALS,POSTAGE,0.077,1.000,1.292
38,RED RETROSPOT PICNIC BAG,POSTAGE,0.069,0.964,1.246
45,SET OF 9 BLACK SKULL BALLOONS,POSTAGE,0.064,0.962,1.243


69 rules with confidence of 96–100%, and they are worthless: French customers buy postage whatever else they buy. Lift shows it. Because postage is in 77% of baskets, no rule ending in postage can have a lift above 1 / 0.77 = 1.30, and these sit right at that ceiling. Compare the skull plates and cups at lift 14. **Always sort by lift, not confidence**, and remove items that are in nearly every basket.

## 6. Pitfall: rare items give spurious rules

Lower the minimum support and the rule count explodes, and the top of the list fills with pairs seen in only two or three baskets.

In [10]:
for s in [0.05, 0.03, 0.02]:
    fi = apriori(france, min_support=s, use_colnames=True)
    r = association_rules(fi, metric="lift", min_threshold=1.0)
    top = r.sort_values("lift", ascending=False).head(10)
    print(f"min_support {s}: {len(r):>5} rules; top-10 by lift seen in as few as "
          f"{(top['support'] * len(france)).min():.0f} baskets")

min_support 0.05:    86 rules; top-10 by lift seen in as few as 20 baskets
min_support 0.03:  1186 rules; top-10 by lift seen in as few as 12 baskets
min_support 0.02:  6078 rules; top-10 by lift seen in as few as 8 baskets


A pair seen in eight baskets can easily have a lift of 20 by coincidence. Keep a sensible minimum support, and check that a rule holds on a different period or country before acting on it (exercise 3).

## Quiz answers

1. Support of {A, B} is the share of baskets that contain both A and B.
2. Confidence(A → B) = support(A, B) / support(A): of the baskets with A, the share that also have B.
3. Lift = confidence(A → B) / support(B). Lift 1: no link. Above 1: bought together more than by chance. Below 1: less.
4. If an itemset is infrequent, every larger itemset containing it is also infrequent, so Apriori never needs to count them.
5. A very popular item such as postage gives rules with high confidence from almost anything, yet their lift cannot rise much above 1 (at most 1 / support of the item). Sort by lift and remove near-universal items.

**Next:** [06.10 Recommender systems](../10-recommenders/)